# CNN Batik Nusantara
**Dataset:** `hendryhb/cnn-batik-nusantara` (Kaggle)

Jalankan setiap sel secara berurutan dari atas ke bawah.

## SEL 1 — Instalasi & Persiapan Direktori

In [ ]:
# Instal library yang dibutuhkan
!pip install -q kaggle seaborn scikit-learn

# Buat struktur direktori proyek
!mkdir -p src data/raw data/processed saved_models outputs

print('Direktori berhasil dibuat!')

## SEL 2 — Upload Kaggle API Key

In [ ]:
from google.colab import files

print('Upload file kaggle.json dari komputer Anda...')
files.upload()

# Pindahkan ke lokasi yang dikenali Kaggle CLI
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print('Kaggle API key berhasil dikonfigurasi!')

## SEL 3 — Tulis `src/__init__.py`

In [ ]:
%%writefile src/__init__.py
# Menandai 'src' sebagai Python package

## SEL 4 — Tulis `src/config.py`

In [ ]:
%%writefile src/config.py
"""
config.py — Konfigurasi pusat proyek CNN Batik Nusantara.
Berisi semua hyperparameter, path dataset, dan konstanta pelatihan.
"""

import os

# ---------------------------------------------------------------------------
# Hyperparameter Model
# ---------------------------------------------------------------------------
IMG_HEIGHT = 150          # Tinggi gambar setelah di-resize
IMG_WIDTH  = 150          # Lebar gambar setelah di-resize
IMG_SIZE   = (IMG_HEIGHT, IMG_WIDTH)
CHANNELS   = 3            # Jumlah channel warna (RGB)
INPUT_SHAPE = (IMG_HEIGHT, IMG_WIDTH, CHANNELS)

BATCH_SIZE    = 32        # Jumlah sampel per batch
EPOCHS        = 30        # Jumlah epoch pelatihan
LEARNING_RATE = 1e-3      # Learning rate Adam
DROPOUT_RATE  = 0.5       # Dropout rate

VALIDATION_SPLIT = 0.2    # 20% data untuk validasi
RANDOM_SEED      = 42     # Seed reproduktibilitas

# ---------------------------------------------------------------------------
# Konfigurasi Dataset Kaggle
# ---------------------------------------------------------------------------
KAGGLE_DATASET = 'hendryhb/cnn-batik-nusantara'
DATASET_NAME   = 'cnn-batik-nusantara'

# ---------------------------------------------------------------------------
# Path Direktori Proyek
# ---------------------------------------------------------------------------
BASE_DIR        = os.getcwd()
DATA_DIR        = os.path.join(BASE_DIR, 'data')
RAW_DIR         = os.path.join(DATA_DIR, 'raw')
PROCESSED_DIR   = os.path.join(DATA_DIR, 'processed')
SAVED_MODELS_DIR = os.path.join(BASE_DIR, 'saved_models')
OUTPUTS_DIR     = os.path.join(BASE_DIR, 'outputs')

# ---------------------------------------------------------------------------
# Path File Output
# ---------------------------------------------------------------------------
MODEL_SAVE_PATH       = os.path.join(SAVED_MODELS_DIR, 'model_batik.keras')
HISTORY_SAVE_PATH     = os.path.join(OUTPUTS_DIR, 'training_history.pkl')
ACCURACY_PLOT_PATH    = os.path.join(OUTPUTS_DIR, 'plot_accuracy.png')
LOSS_PLOT_PATH        = os.path.join(OUTPUTS_DIR, 'plot_loss.png')
CONFUSION_MATRIX_PATH = os.path.join(OUTPUTS_DIR, 'confusion_matrix.png')

## SEL 5 — Tulis `src/data_loader.py`

In [ ]:
%%writefile src/data_loader.py
"""
data_loader.py — Unduh dataset Kaggle dan buat data generator.
"""

import os
import subprocess
import sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from src.config import (
    KAGGLE_DATASET, RAW_DIR, PROCESSED_DIR,
    IMG_HEIGHT, IMG_WIDTH, BATCH_SIZE,
    VALIDATION_SPLIT, RANDOM_SEED
)


def unduh_dataset():
    """Unduh dan ekstrak dataset dari Kaggle. Return path direktori dataset."""
    print('=' * 60)
    print(f'[INFO] Mengunduh dataset: {KAGGLE_DATASET}')
    print('=' * 60)

    os.makedirs(RAW_DIR, exist_ok=True)
    os.makedirs(PROCESSED_DIR, exist_ok=True)

    cmd = [
        'kaggle', 'datasets', 'download',
        '-d', KAGGLE_DATASET,
        '-p', RAW_DIR,
        '--unzip'
    ]
    print(f'[INFO] Perintah: {" ".join(cmd)}')

    try:
        hasil = subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(f'[SUKSES] {hasil.stdout}')
    except subprocess.CalledProcessError as e:
        print(f'[ERROR] {e.stderr}')
        raise

    return _temukan_direktori_dataset(RAW_DIR)


def _temukan_direktori_dataset(direktori_awal):
    """Cari direktori yang berisi subfolder kelas gambar."""
    for root, dirs, _ in os.walk(direktori_awal):
        subdirs = [d for d in dirs if os.path.isdir(os.path.join(root, d))]
        if len(subdirs) > 1:
            print(f'[INFO] Ditemukan {len(subdirs)} kelas di: {root}')
            return root
    print(f'[PERINGATAN] Menggunakan direktori fallback: {direktori_awal}')
    return direktori_awal


def dapatkan_data_generators(direktori_dataset=None):
    """
    Buat ImageDataGenerator untuk training (80%) dan validasi (20%).
    Training dilengkapi augmentasi; validasi hanya normalisasi.

    Returns:
        tuple: (train_gen, val_gen, num_classes, class_names)
    """
    if direktori_dataset is None:
        direktori_dataset = _temukan_direktori_dataset(RAW_DIR)

    print(f'[INFO] Direktori dataset: {direktori_dataset}')
    print(f'[INFO] Ukuran gambar: {IMG_HEIGHT}x{IMG_WIDTH} | Batch: {BATCH_SIZE}')

    # Generator training dengan augmentasi
    train_datagen = ImageDataGenerator(
        rescale=1.0 / 255,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.15,
        zoom_range=0.15,
        horizontal_flip=True,
        fill_mode='nearest',
        validation_split=VALIDATION_SPLIT
    )

    # Generator validasi tanpa augmentasi
    val_datagen = ImageDataGenerator(
        rescale=1.0 / 255,
        validation_split=VALIDATION_SPLIT
    )

    # Flow dari direktori
    train_gen = train_datagen.flow_from_directory(
        direktori_dataset,
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='training',
        seed=RANDOM_SEED,
        shuffle=True
    )

    val_gen = val_datagen.flow_from_directory(
        direktori_dataset,
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='validation',
        seed=RANDOM_SEED,
        shuffle=False
    )

    num_classes = len(train_gen.class_indices)
    class_names = list(train_gen.class_indices.keys())

    print(f'[INFO] Kelas ({num_classes}): {class_names}')
    print(f'[INFO] Sampel training: {train_gen.samples} | validasi: {val_gen.samples}')

    return train_gen, val_gen, num_classes, class_names

## SEL 6 — Tulis `src/model.py`

In [ ]:
%%writefile src/model.py
"""
model.py — Arsitektur CNN untuk klasifikasi batik nusantara.

Arsitektur:
  Blok 1: Conv2D(32)  + BN + MaxPool
  Blok 2: Conv2D(64)  + BN + MaxPool
  Blok 3: Conv2D(128) x2 + BN + MaxPool
  Head:   Flatten -> Dense(256,ReLU) -> Dropout(0.5) -> Dense(N,Softmax)
"""

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten,
    Dense, Dropout, BatchNormalization, Input
)


def bangun_model(input_shape, num_classes):
    """
    Bangun dan kembalikan model CNN Sequential.

    Args:
        input_shape (tuple): Misal (150, 150, 3).
        num_classes (int): Jumlah kelas batik.

    Returns:
        tf.keras.Sequential: Model yang belum dikompilasi.
    """
    print(f'[INFO] Membangun model — Input: {input_shape} | Kelas: {num_classes}')

    model = Sequential(name='CNN_Batik_Nusantara')
    model.add(Input(shape=input_shape))

    # Blok 1 — fitur dasar (tepi, tekstur)
    model.add(Conv2D(32, (3,3), padding='same', activation='relu', name='conv1'))
    model.add(BatchNormalization(name='bn1'))
    model.add(MaxPooling2D((2,2), name='pool1'))

    # Blok 2 — fitur menengah (pola geometris batik)
    model.add(Conv2D(64, (3,3), padding='same', activation='relu', name='conv2'))
    model.add(BatchNormalization(name='bn2'))
    model.add(MaxPooling2D((2,2), name='pool2'))

    # Blok 3 — fitur kompleks (motif batik spesifik)
    model.add(Conv2D(128, (3,3), padding='same', activation='relu', name='conv3_1'))
    model.add(Conv2D(128, (3,3), padding='same', activation='relu', name='conv3_2'))
    model.add(BatchNormalization(name='bn3'))
    model.add(MaxPooling2D((2,2), name='pool3'))

    # Head Classifier
    model.add(Flatten(name='flatten'))
    model.add(Dense(256, activation='relu', name='dense1'))
    model.add(Dropout(0.5, name='dropout'))   # Regularisasi: matikan 50% neuron
    model.add(Dense(num_classes, activation='softmax', name='output'))

    model.summary()
    return model


# Alias bahasa Inggris untuk kompatibilitas
build_model = bangun_model

## SEL 7 — Tulis `src/train.py`

In [ ]:
%%writefile src/train.py
"""
train.py — Skrip utama pelatihan CNN Batik Nusantara.

Jalankan dengan: !python src/train.py
"""

import os
import sys
import pickle
import tensorflow as tf
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
)

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from src.config import (
    INPUT_SHAPE, EPOCHS, LEARNING_RATE,
    MODEL_SAVE_PATH, HISTORY_SAVE_PATH,
    SAVED_MODELS_DIR, OUTPUTS_DIR
)
from src.data_loader import unduh_dataset, dapatkan_data_generators
from src.model import bangun_model


def main():
    print('\n' + '='*60)
    print('    CNN BATIK NUSANTARA - PELATIHAN MODEL')
    print('='*60)

    # Cek ketersediaan GPU
    gpus = tf.config.list_physical_devices('GPU')
    print(f'[INFO] GPU: {len(gpus)} unit ditemukan' if gpus else '[PERINGATAN] Tidak ada GPU — menggunakan CPU.')

    # 1. Unduh dataset
    print('\n[LANGKAH 1] Mengunduh dataset...')
    dir_dataset = unduh_dataset()

    # 2. Buat data generator
    print('\n[LANGKAH 2] Mempersiapkan data generator...')
    train_gen, val_gen, num_classes, class_names = dapatkan_data_generators(dir_dataset)

    # 3. Bangun model
    print('\n[LANGKAH 3] Membangun model CNN...')
    model = bangun_model(INPUT_SHAPE, num_classes)

    # 4. Kompilasi
    print('\n[LANGKAH 4] Mengkompilasi model...')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    print(f'[INFO] Optimizer: Adam (lr={LEARNING_RATE}) | Loss: Categorical Crossentropy')

    # 5. Callbacks
    os.makedirs(SAVED_MODELS_DIR, exist_ok=True)
    os.makedirs(OUTPUTS_DIR, exist_ok=True)

    callbacks = [
        # Simpan model terbaik berdasarkan val_accuracy
        ModelCheckpoint(
            filepath=MODEL_SAVE_PATH,
            monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
        ),
        # Hentikan lebih awal jika val_loss tidak membaik
        EarlyStopping(
            monitor='val_loss', patience=7,
            restore_best_weights=True, verbose=1
        ),
        # Kurangi learning rate saat plateau
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=3, min_lr=1e-7, verbose=1
        ),
    ]

    # 6. Hitung steps
    steps_per_epoch  = max(1, train_gen.samples // train_gen.batch_size)
    validation_steps = max(1, val_gen.samples  // val_gen.batch_size)

    # 7. Latih model
    print('\n[LANGKAH 5] Memulai pelatihan...')
    history = model.fit(
        train_gen,
        steps_per_epoch=steps_per_epoch,
        epochs=EPOCHS,
        validation_data=val_gen,
        validation_steps=validation_steps,
        callbacks=callbacks,
        verbose=1
    )

    # 8. Simpan histori (pickle) dan nama kelas
    with open(HISTORY_SAVE_PATH, 'wb') as f:
        pickle.dump(history.history, f)
    print(f'[SUKSES] Histori disimpan: {HISTORY_SAVE_PATH}')

    with open(os.path.join(OUTPUTS_DIR, 'class_names.pkl'), 'wb') as f:
        pickle.dump(class_names, f)

    # Ringkasan
    acc = history.history['accuracy'][-1]
    val_acc = history.history['val_accuracy'][-1]
    print('\n' + '='*60)
    print(f'[HASIL] Training Accuracy : {acc:.4f} ({acc*100:.2f}%)')
    print(f'[HASIL] Validasi Accuracy : {val_acc:.4f} ({val_acc*100:.2f}%)')
    print(f'[HASIL] Model disimpan    : {MODEL_SAVE_PATH}')
    print('='*60)
    print('[INFO] Selesai! Jalankan evaluate.py untuk evaluasi.')


if __name__ == '__main__':
    main()

## SEL 8 — Tulis `src/evaluate.py`

In [ ]:
%%writefile src/evaluate.py
"""
evaluate.py — Evaluasi model & visualisasi hasil.

Jalankan dengan: !python src/evaluate.py
"""

import os
import sys
import pickle
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Backend non-interaktif untuk Colab
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from src.config import (
    MODEL_SAVE_PATH, HISTORY_SAVE_PATH,
    ACCURACY_PLOT_PATH, LOSS_PLOT_PATH,
    CONFUSION_MATRIX_PATH, OUTPUTS_DIR
)
from src.data_loader import unduh_dataset, dapatkan_data_generators


def muat_model(path):
    """Muat model Keras dari file .keras."""
    if not os.path.exists(path):
        raise FileNotFoundError(f'[ERROR] Model tidak ditemukan: {path}. Jalankan train.py dulu!')
    model = tf.keras.models.load_model(path)
    print(f'[SUKSES] Model dimuat dari: {path}')
    model.summary()
    return model


def plot_akurasi(hist, path):
    """Grafik Training vs Validation Accuracy."""
    epochs = range(1, len(hist['accuracy']) + 1)
    best_e = hist['val_accuracy'].index(max(hist['val_accuracy'])) + 1

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(epochs, hist['accuracy'],   'b-o', lw=2, ms=4, label='Training Accuracy')
    ax.plot(epochs, hist['val_accuracy'],'r-o', lw=2, ms=4, label='Validation Accuracy')
    ax.axvline(best_e, color='green', ls='--', alpha=0.6, label=f'Best Epoch: {best_e}')
    ax.set_title('Kurva Akurasi Training vs Validasi', fontsize=14, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Akurasi')
    ax.set_ylim([0, 1.05]); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close()
    print(f'[SUKSES] Grafik akurasi: {path}')


def plot_loss(hist, path):
    """Grafik Training vs Validation Loss."""
    epochs = range(1, len(hist['loss']) + 1)
    best_e = hist['val_loss'].index(min(hist['val_loss'])) + 1

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(epochs, hist['loss'],     'b-o', lw=2, ms=4, label='Training Loss')
    ax.plot(epochs, hist['val_loss'], 'r-o', lw=2, ms=4, label='Validation Loss')
    ax.axvline(best_e, color='green', ls='--', alpha=0.6, label=f'Best Epoch: {best_e}')
    ax.set_title('Kurva Loss Training vs Validasi', fontsize=14, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close()
    print(f'[SUKSES] Grafik loss: {path}')


def evaluasi_dan_laporan(model, val_gen, class_names):
    """Evaluasi model, tampilkan Classification Report."""
    print('\n[INFO] Evaluasi Keras:')
    val_loss, val_acc = model.evaluate(val_gen, verbose=1)
    print(f'[HASIL] Loss: {val_loss:.4f} | Accuracy: {val_acc:.4f} ({val_acc*100:.2f}%)')

    val_gen.reset()
    y_pred = np.argmax(model.predict(val_gen, verbose=1), axis=1)
    y_true = val_gen.classes

    laporan = classification_report(y_true, y_pred, target_names=class_names, digits=4)
    print('\n' + '='*60 + '\n    CLASSIFICATION REPORT\n' + '='*60)
    print(laporan)

    path_laporan = os.path.join(OUTPUTS_DIR, 'classification_report.txt')
    with open(path_laporan, 'w', encoding='utf-8') as f:
        f.write('CLASSIFICATION REPORT - CNN BATIK NUSANTARA\n')
        f.write('='*60 + '\n' + laporan)
    print(f'[SUKSES] Laporan disimpan: {path_laporan}')

    return y_true, y_pred


def plot_confusion_matrix(y_true, y_pred, class_names, path):
    """Buat dan simpan Confusion Matrix (normalisasi per baris)."""
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    n = len(class_names)
    size = max(10, n * 0.8)
    fig, ax = plt.subplots(figsize=(size, size * 0.85))
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, linewidths=0.5)
    ax.set_title('Confusion Matrix (Normalisasi)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Prediksi', fontsize=12); ax.set_ylabel('Label Asli', fontsize=12)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
    plt.tight_layout(); plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close()
    print(f'[SUKSES] Confusion Matrix: {path}')


def main():
    print('\n' + '='*60)
    print('    CNN BATIK NUSANTARA - EVALUASI MODEL')
    print('='*60)

    os.makedirs(OUTPUTS_DIR, exist_ok=True)

    # 1. Muat model
    model = muat_model(MODEL_SAVE_PATH)

    # 2. Muat dan plot histori
    if os.path.exists(HISTORY_SAVE_PATH):
        with open(HISTORY_SAVE_PATH, 'rb') as f:
            hist = pickle.load(f)
        plot_akurasi(hist, ACCURACY_PLOT_PATH)
        plot_loss(hist, LOSS_PLOT_PATH)
    else:
        print('[PERINGATAN] Histori tidak ditemukan — lewati visualisasi.')

    # 3. Siapkan data validasi
    class_names_path = os.path.join(OUTPUTS_DIR, 'class_names.pkl')
    dir_dataset = unduh_dataset()
    _, val_gen, _, class_names = dapatkan_data_generators(dir_dataset)

    # Gunakan nama kelas dari pickle jika tersedia
    if os.path.exists(class_names_path):
        with open(class_names_path, 'rb') as f:
            class_names = pickle.load(f)

    # 4. Evaluasi & Laporan
    y_true, y_pred = evaluasi_dan_laporan(model, val_gen, class_names)

    # 5. Confusion Matrix
    plot_confusion_matrix(y_true, y_pred, class_names, CONFUSION_MATRIX_PATH)

    # Ringkasan
    akurasi = accuracy_score(y_true, y_pred)
    print('\n' + '='*60)
    print(f'[HASIL] Akurasi Keseluruhan: {akurasi:.4f} ({akurasi*100:.2f}%)')
    print('\n[OUTPUT] File tersimpan di folder outputs/:')
    for p in [ACCURACY_PLOT_PATH, LOSS_PLOT_PATH, CONFUSION_MATRIX_PATH]:
        print(f'  - {p}')
    print('='*60)


if __name__ == '__main__':
    main()

## SEL 9 — Verifikasi Struktur File
Jalankan sel ini untuk memastikan semua file berhasil dibuat.

In [ ]:
import os

files_diharapkan = [
    'src/__init__.py',
    'src/config.py',
    'src/data_loader.py',
    'src/model.py',
    'src/train.py',
    'src/evaluate.py',
]

print('✅ Verifikasi File Proyek:')
semua_ok = True
for f in files_diharapkan:
    ada = os.path.exists(f)
    status = '✓ ADA' if ada else '✗ TIDAK ADA'
    print(f'  {status} — {f}')
    if not ada:
        semua_ok = False

print()
if semua_ok:
    print('🎉 Semua file siap! Lanjutkan ke SEL 10 untuk melatih model.')
else:
    print('⚠️  Ada file yang hilang. Jalankan ulang sel yang bersangkutan.')

---
## SEL 10 — 🚀 Jalankan Pelatihan Model

> **Pastikan GPU aktif:** `Runtime → Change runtime type → GPU` sebelum menjalankan sel ini.

Proses ini akan:
1. Mengunduh dataset dari Kaggle
2. Melatih model CNN selama maks. 30 epoch
3. Menyimpan model terbaik ke `saved_models/model_batik.keras`
4. Menyimpan histori ke `outputs/training_history.pkl`

In [ ]:
!python src/train.py

---
## SEL 11 — 📊 Jalankan Evaluasi & Visualisasi

Proses ini akan:
1. Memuat model dari `saved_models/model_batik.keras`
2. Membuat grafik Akurasi & Loss → `outputs/`
3. Mencetak Classification Report (Precision, Recall, F1)
4. Menyimpan Confusion Matrix → `outputs/confusion_matrix.png`

In [ ]:
!python src/evaluate.py

## SEL 12 — 🖼️ Tampilkan Output Visual di Notebook

In [ ]:
from IPython.display import Image, display

print('=== Grafik Akurasi ===')
display(Image('outputs/plot_accuracy.png'))

print('\n=== Grafik Loss ===')
display(Image('outputs/plot_loss.png'))

print('\n=== Confusion Matrix ===')
display(Image('outputs/confusion_matrix.png'))

## SEL 13 — 💾 Download Semua Output (Opsional)

In [ ]:
import shutil
from google.colab import files

# Zip semua output
shutil.make_archive('hasil_cnn_batik', 'zip', '.', 'outputs')
shutil.make_archive('model_batik', 'zip', '.', 'saved_models')

# Download ke komputer lokal
files.download('hasil_cnn_batik.zip')
files.download('model_batik.zip')
print('Download dimulai!')